[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%204/search_hackathon.ipynb)

# The Retrieval Hackathon — Meaning vs. Words

**Week 4, Lecture 7.**

**The challenge.** Build the best search engine for this corpus of ~800 passages. Beat the keyword baseline, then beat each other. Same corpus, same queries, same metric for every team — *how* you build the retriever is wide open.

**The metric.** For each query, your retriever ranks the passages; we check where the correct one lands. **MRR@10** (higher = the right passage sits near the top) is the headline score, alongside **Recall@5** and your **index build time** (the cost axis).

**Two rounds.** The **main** queries are literal — keyword search is strong here. The **curveball** queries are paraphrased (different words, same meaning) — that's where semantic search earns its keep. Your row reports both.

## Section 0 — Setup: load the data and the scoring (everyone runs this first)

In [ ]:
import json, time, urllib.request
import numpy as np

BASE = "https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%204"
def load_jsonl(name):
    try:
        raw = urllib.request.urlopen(f"{BASE}/{name}").read().decode()   # from the course repo
    except Exception:
        raw = open(name).read()                                          # local fallback
    return [json.loads(l) for l in raw.splitlines() if l.strip()]

corpus  = [r["text"] for r in sorted(load_jsonl("corpus.jsonl"), key=lambda r: r["id"])]
main    = load_jsonl("queries.jsonl")       # literal questions
curve   = load_jsonl("curveball.jsonl")     # paraphrased questions (the plot twist)
print("corpus:", len(corpus), " main queries:", len(main), " curveball:", len(curve))

# --- scoring: your retriever is a function  query(str) -> list of doc ids, best first ---
def score(retriever, queries):
    r1 = r5 = mrr = 0
    for q in queries:
        ranked = retriever(q["query"])
        rank = next((j+1 for j,idx in enumerate(ranked) if idx == q["gold_id"]), 999)
        r1  += rank == 1
        r5  += rank <= 5
        mrr += (1.0/rank) if rank <= 10 else 0
    n = len(queries)
    return {"r@1": r1/n, "r@5": r5/n, "mrr": mrr/n}

def evaluate(retriever):
    return score(retriever, main), score(retriever, curve)

def submit(team, method, retriever, build_seconds):
    m, c = evaluate(retriever)
    print("\n" + "="*58)
    print("  LEADERBOARD ROW")
    print("="*58)
    print(f"  Team            : {team}")
    print(f"  Method          : {method}")
    print(f"  Main MRR@10     : {m['mrr']:.3f}")
    print(f"  Curveball MRR   : {c['mrr']:.3f}   (paraphrase queries)")
    print(f"  Main Recall@5   : {m['r@5']:.2f}")
    print(f"  Index build (s) : {build_seconds:.1f}")
    print("-"*58)
    print("  Paste this tab-separated line into the leaderboard:")
    print(f"  {team}\t{method}\t{m['mrr']:.3f}\t{c['mrr']:.3f}\t{m['r@5']:.2f}\t{build_seconds:.1f}")

## Your baseline to beat: TF-IDF keyword search (given)

In [ ]:
# Baseline: TF-IDF keyword search. This is the interface your retriever must match:
#   a function that takes a query string and returns doc ids ranked best-first.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import time

t0 = time.time()
tfidf = TfidfVectorizer(stop_words="english")
doc_vectors = tfidf.fit_transform(corpus)          # <- this is the 'index'
build_seconds = time.time() - t0

def tfidf_search(query):
    sims = cosine_similarity(tfidf.transform([query]), doc_vectors)[0]
    return list(np.argsort(-sims))

m, c = evaluate(tfidf_search)
print(f"TF-IDF baseline  ->  main MRR {m['mrr']:.3f}   |   curveball MRR {c['mrr']:.3f}")
# When you're ready, post it:  submit('Team ?', 'TF-IDF baseline', tfidf_search, build_seconds)

## Build your retriever

Beat the baseline on the leaderboard. Pick any approach — the hints below are starting points, not limits.

In [ ]:
# ---- YOUR TURN: build a retriever that beats the baseline ----
# Match the same interface: a function  query(str) -> list of doc ids, best first.
# Time your index build so you can report the cost.
#
# Ideas (each is a pip install away in Colab):
#   BM25     : !pip -q install rank_bm25 ; from rank_bm25 import BM25Okapi
#   Dense    : !pip -q install sentence-transformers
#              m = SentenceTransformer("all-MiniLM-L6-v2")
#              index = m.encode(corpus, normalize_embeddings=True)   # encode once
#              rank by cosine similarity to the query embedding
#   Bigger   : "all-mpnet-base-v2" (slower, usually better)
#   Hybrid   : fuse BM25 + dense rankings (reciprocal rank fusion)
#   Re-rank  : take BM25's top 50, reorder with a cross-encoder
#
# t0 = time.time()
# ... build your index ...
# build_seconds = time.time() - t0
# def my_search(query):
#     ...
#     return ranked_ids
# submit('Team X', 'BM25', my_search, build_seconds)


## Post to the leaderboard, then we read it together

Paste your tab-separated row into the shared class leaderboard:

**[Retrieval Hackathon Leaderboard](https://docs.google.com/spreadsheets/d/18asBX7QYPLyA5x2jGoq2tswkoY-dPTxjWdKHBmBkNw8/edit)**

**Bring back an answer to:**
1. Where did keyword search win, and where did it fall apart?
2. How much did the curveball (paraphrase) round change the ranking versus the main round?
3. Where does your retriever sit on the quality-vs-cost frontier?

*ISBA 2411 - Natural Language Processing & AI - Summer 2026*